In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import joblib
import os

In [2]:
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")
MODELS_DIR = os.path.join(PROJECT_DIR, "models")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

In [3]:
COLUMN_MAP = {
    "Amount Requested": "loan_amnt",
    "Debt-To-Income Ratio": "dti",
    "Employment Length": "emp_length",
    "Risk_Score": "risk_score",
    "Loan Title": "purpose",
    "State": "addr_state",
    "Application Date": "app_date",
}

In [4]:
print("Loading raw CSVs...")
accepted = pd.read_csv(os.path.join(RAW_DIR, "accepted_2007_to_2018Q4.csv"), low_memory=False)
rejected = pd.read_csv(os.path.join(RAW_DIR, "rejected_2007_to_2018Q4.csv"), low_memory=False)
rejected = rejected.rename(columns=COLUMN_MAP)

Loading raw CSVs...


In [5]:
print("Accepted shape:", accepted.shape)
print("Rejected shape:", rejected.shape)

Accepted shape: (2260701, 151)
Rejected shape: (27648741, 9)


In [6]:
print("\nBuilding approval dataset...")

# Only these fields are recorded the same way in both files
APPROVAL_FEATURES = ["loan_amnt", "dti", "risk_score"]

rej_common = rejected[APPROVAL_FEATURES + ["addr_state", "app_date"]].dropna(
    subset=["risk_score", "dti"]).copy()
rej_common["approved"] = 0

acc_common = accepted[["loan_amnt", "dti", "fico_range_low", "fico_range_high",
                       "addr_state", "issue_d"]].copy()
acc_common["risk_score"] = (acc_common["fico_range_low"] + acc_common["fico_range_high"]) / 2
acc_common = acc_common.rename(columns={"issue_d": "app_date"})
acc_common = acc_common.drop(columns=["fico_range_low", "fico_range_high"])
acc_common = acc_common.dropna(subset=["risk_score", "dti"])
acc_common["approved"] = 1

# Compare like with like: accepted loans only from the period in which
# rejected applications also carry a credit score
rej_dates = pd.to_datetime(rej_common["app_date"], errors="coerce")
cutoff = rej_dates.max()
assert pd.notna(cutoff), "Could not read the rejected 'Application Date' column"
acc_dates = pd.to_datetime(acc_common["app_date"], format="%b-%Y", errors="coerce")
acc_common = acc_common[acc_dates <= cutoff].copy()
print("Credit scores in rejected file run up to:", cutoff.date())

# Convert numeric columns (text like "12.5%" -> 12.5), whatever the dtype
for df_ in (acc_common, rej_common):
    df_["dti"] = pd.to_numeric(
        df_["dti"].astype(str).str.replace("%", "", regex=False), errors="coerce")
    df_["risk_score"] = pd.to_numeric(df_["risk_score"], errors="coerce")
    df_["loan_amnt"] = pd.to_numeric(df_["loan_amnt"], errors="coerce")

# Only compare requests inside the range LendingClub actually funds
max_funded = acc_common["loan_amnt"].max()
rej_common = rej_common[rej_common["loan_amnt"] <= max_funded]
print("Largest funded loan:", max_funded)
print("Accepted kept:", len(acc_common), "| Rejected kept:", len(rej_common))

approval_df = pd.concat([acc_common, rej_common], ignore_index=True)
approval_df = approval_df.dropna(subset=["loan_amnt", "dti", "risk_score"])
approval_df = approval_df[(approval_df["dti"] >= 0) & (approval_df["dti"] <= 100)]


Building approval dataset...
Credit scores in rejected file run up to: 2018-12-31
Largest funded loan: 40000.0
Accepted kept: 2258957 | Rejected kept: 9147483


In [7]:
approved_rows = approval_df[approval_df["approved"] == 1]
rejected_rows = approval_df[approval_df["approved"] == 0]

n = min(len(approved_rows), len(rejected_rows))
approval_balanced = pd.concat([
    resample(approved_rows, n_samples=n, replace=False, random_state=42),
    resample(rejected_rows, n_samples=n, replace=False, random_state=42),
]).reset_index(drop=True)

print(approval_balanced["approved"].value_counts())

approved
1    2256394
0    2256394
Name: count, dtype: int64


In [8]:
approval_df["dti"] = (
    approval_df["dti"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

approval_df["dti"] = pd.to_numeric(
    approval_df["dti"],
    errors="coerce"
)

In [9]:
# State is not a model feature. Keep it aside only for the fairness audit.
audit_state = approval_balanced["addr_state"].astype(str)
X_app = approval_balanced[APPROVAL_FEATURES]
y_app = approval_balanced["approved"]

X_train_app, X_test_app, y_train_app, y_test_app, _, state_test = train_test_split(
    X_app, y_app, audit_state, test_size=0.2, stratify=y_app, random_state=42)

X_train_app.assign(approved=y_train_app).to_csv(
    os.path.join(PROCESSED_DIR, "approval_train.csv"), index=False)
X_test_app.assign(approved=y_test_app).to_csv(
    os.path.join(PROCESSED_DIR, "approval_test.csv"), index=False)
state_test.to_frame("addr_state").to_csv(
    os.path.join(PROCESSED_DIR, "approval_test_state.csv"), index=False)

print("Approval dataset saved. Train:", X_train_app.shape, "Test:", X_test_app.shape)

Approval dataset saved. Train: (3610230, 3) Test: (902558, 3)


In [10]:
print("\nBuilding default dataset...")
default_df = accepted[accepted["loan_status"].isin(
    ["Fully Paid", "Charged Off", "Default"])].copy()
default_df["target"] = default_df["loan_status"].apply(
    lambda x: 1 if x in ["Charged Off", "Default"] else 0)

missing_pct = default_df.isnull().mean()
default_df = default_df.drop(columns=missing_pct[missing_pct > 0.5].index)



Building default dataset...


In [11]:
drop_cols = [
    "id", "member_id", "url", "desc", "emp_title", "loan_status",

    # Post-loan / target leakage
    "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low",
    "hardship_flag", "debt_settlement_flag",

    # Location proxies and free text
    "zip_code", "addr_state", "title",

    # Near-duplicates of other features
    "funded_amnt", "funded_amnt_inv", "sub_grade",
]
default_df = default_df.drop(columns=drop_cols, errors="ignore")

# Dates -> credit history length; two FICO columns -> one score
issue = pd.to_datetime(default_df["issue_d"], format="%b-%Y", errors="coerce")
earliest = pd.to_datetime(default_df["earliest_cr_line"], format="%b-%Y", errors="coerce")
default_df["credit_history_years"] = (issue - earliest).dt.days / 365.25
default_df["fico_score"] = (default_df["fico_range_low"] + default_df["fico_range_high"]) / 2
default_df = default_df.drop(columns=["issue_d", "earliest_cr_line",
                                      "fico_range_low", "fico_range_high"])

print("\nRemaining default features:")
print(default_df.columns.tolist())
print("\nNumber of features:", len(default_df.columns))

# Fill missing values (works on both pandas 2 and 3)
for col in default_df.columns:
    if col == "target":
        continue
    if pd.api.types.is_numeric_dtype(default_df[col]):
        default_df[col] = default_df[col].fillna(default_df[col].median())
    else:
        default_df[col] = default_df[col].fillna(default_df[col].mode()[0])


Remaining default features:
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'pymnt_plan', 'purpose', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'policy_code', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec

In [12]:
default_encoders = {}

categorical_cols = [c for c in default_df.columns
                    if c != "target" and not pd.api.types.is_numeric_dtype(default_df[c])]
print("Categorical columns:", categorical_cols)

for col in categorical_cols:
    le = LabelEncoder()
    default_df[col] = le.fit_transform(default_df[col].astype(str))
    default_encoders[col] = le

joblib.dump(default_encoders, os.path.join(MODELS_DIR, "default_encoders.pkl"))

# Separate features and target
X_def = default_df.drop(columns="target")
y_def = default_df["target"]

# Train-test split
X_train_def, X_test_def, y_train_def, y_test_def = train_test_split(
    X_def, y_def, test_size=0.2, stratify=y_def, random_state=42)

# Scale numeric columns AFTER splitting
numeric_cols = X_train_def.select_dtypes(include="number").columns

scaler = StandardScaler()
X_train_def[numeric_cols] = scaler.fit_transform(X_train_def[numeric_cols])
X_test_def[numeric_cols] = scaler.transform(X_test_def[numeric_cols])

joblib.dump(scaler, os.path.join(MODELS_DIR, "default_scaler.pkl"))

Categorical columns: ['term', 'grade', 'emp_length', 'home_ownership', 'verification_status', 'pymnt_plan', 'purpose', 'initial_list_status', 'application_type', 'disbursement_method']


['c:\\Users\\T9926\\Desktop\\loan_detector\\models\\default_scaler.pkl']

In [13]:
print("\nBefore SMOTE:")
print(y_train_def.value_counts())

MAX_TRAIN_SIZE = 50000

if len(X_train_def) > MAX_TRAIN_SIZE:

    X_train_sample, _, y_train_sample, _ = train_test_split(
        X_train_def,
        y_train_def,
        train_size=MAX_TRAIN_SIZE,
        stratify=y_train_def,
        random_state=42
    )

else:
    X_train_sample = X_train_def
    y_train_sample = y_train_def

print("Training data used for SMOTE:", X_train_sample.shape)

# Apply SMOTE
smote = SMOTE(random_state=42)

X_train_def_res, y_train_def_res = smote.fit_resample(
    X_train_sample,
    y_train_sample
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_def_res).value_counts())

# Save balanced training data
X_train_def_res.assign(
    target=y_train_def_res
).to_csv(
    os.path.join(PROCESSED_DIR, "default_train.csv"),
    index=False
)

# Save original test data WITHOUT SMOTE
X_test_def.assign(
    target=y_test_def
).to_csv(
    os.path.join(PROCESSED_DIR, "default_test.csv"),
    index=False
)

print(
    "\nDefault dataset saved.",
    "Train after SMOTE:", X_train_def_res.shape,
    "Test:", X_test_def.shape
)

print("\nDone. Processed files are in data/processed/, encoders/scaler in models/.")


Before SMOTE:
target
0    861401
1    214879
Name: count, dtype: int64
Training data used for SMOTE: (50000, 65)

After SMOTE:
target
0    40018
1    40018
Name: count, dtype: int64

Default dataset saved. Train after SMOTE: (80036, 65) Test: (269070, 65)

Done. Processed files are in data/processed/, encoders/scaler in models/.
